# SFT: QLoRA (Kaggle GPU)

Qwen2.5-0.5B base modelini QLoRA bilan o'qitadi, 68 ta test savolida baholaydi va adapterni yuklab olish uchun zip qiladi.

Boshlashdan oldin o'ng tomondagi **Settings** bo'limida:

- **Accelerator → GPU T4 x2** (yoki GPU P100)
- **Internet → On** (buning uchun Kaggle akkaunti telefon raqami bilan tasdiqlangan bo'lishi kerak)

Keyin **Run All** ni bosing.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

Kod va dataset GitHub'dan, model fayli (942 MB) Hugging Face'dan olinadi.

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

!rm -f Qwen2.5-0.5B/model.safetensors
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-0.5B", local_dir="Qwen2.5-0.5B")
!ls -lh Qwen2.5-0.5B/model.safetensors
!wc -l sft/train.jsonl sft/val.jsonl

## 3. QLoRA bilan o'qitish

Sozlamalar: 10 epoch, rank 64. Base model 4-bit ko'rinishda yuklanadi; faqat validation loss eng yaxshi bo'lgan epoch saqlanadi.

Chiqishning boshida `QLoRA: ... ta qatlam 4-bit` degan qator bo'lishi kerak. Bo'lmasa, bu haqiqiy QLoRA emas.

In [ ]:
!python sft/qlora_kaggle/train_qlora.py --epochs 10 --rank 64

## 4. Adapterni zip qilish

Avval adapter saqlab olinadi, baholash keyin: sessiya uzilib qolsa ham adapter yo'qolmaydi.

In [ ]:
!head -c 150 chat_ui/adapters/sft-qlora/training_log.json
!rm -f /kaggle/working/qlora_adapter.zip
!zip -q -r /kaggle/working/qlora_adapter.zip chat_ui/adapters/sft-qlora
!ls -lh /kaggle/working/qlora_adapter.zip

## 5. Baholash: 68 ta test savoli

Base model: 5/68. CPU'dagi LoRA (rank 16, 3 epoch): 32/68.

In [ ]:
!python eval/run_eval.py --model sft-qlora | tail -12
!zip -q /kaggle/working/qlora_adapter.zip eval/results/sft-qlora.json

## 6. Yuklab olish

Quyidagi havolani bosing, yoki o'ng tomondagi **Output** bo'limidan `qlora_adapter.zip` ni yuklab oling. Faylni loyihaning `sft` papkasiga qo'ying.

In [ ]:
%cd /kaggle/working
from IPython.display import FileLink
FileLink("qlora_adapter.zip")